# Vertically stacked heterodyne spectra, varying one parameter
Stacks heterodyne spectra of N_p-particle x-y arrays (LAM) while one parameter is varied per panel: the coupling g, the ratio g_y/g_x, the detuning Δ, the cavity linewidth κ, or a set of random couplings with a given RMS value (seeded). The frequency models are pairs, equal spacing and a custom four-band comb.

Runtime: about 2 min (most of it in the "All possibilities" section, which draws 60 figures).

# START

# Define setup and plotting functions

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from levisim import *

set_plt_rcParams_to_xmgrace_style()

# seeded, so that the random RMS-coupling samples (and hence the figures) are reproducible
rng = np.random.default_rng(2025)

def sample_with_RMS(num_samples, lower_bound, upper_bound, desired_RMS, max_iter=1000, tol=1e-6):

    # check if R is feasible
    RMS_min = max(0, min(abs(lower_bound), abs(upper_bound)) if lower_bound * upper_bound <= 0 else min(lower_bound, upper_bound))
    RMS_max = max(abs(lower_bound), abs(upper_bound))
    if desired_RMS < RMS_min or desired_RMS > RMS_max:
        raise ValueError(f'{desired_RMS=} is not achievable in the range [{lower_bound}, {upper_bound}]')

    # Generate initial random numbers in [a, b]
    initial_random = rng.uniform(lower_bound, upper_bound, num_samples)

    for _ in range(max_iter):
        # compute current RMS and rescale
        current_RMS = np.sqrt(np.mean(initial_random**2))
        initial_random = initial_random * (desired_RMS / current_RMS)

        # clip values outside [a, b]
        initial_random = np.clip(initial_random, lower_bound, upper_bound)

        # check if RMS is close enough
        if np.abs(np.sqrt(np.mean(initial_random**2)) - desired_RMS) < tol:
            return initial_random

    raise RuntimeError('Failed to converge')

num_samples = 10
lower_bound, upper_bound = 5, 100
desired_RMS = 40

samples = sample_with_RMS(num_samples, lower_bound, upper_bound, desired_RMS)
print('Samples:', samples)
print('Desired RMS:', desired_RMS)
print('Actual RMS:', np.sqrt(np.mean(samples**2)))


def make_Ω_model_dict(Ω_model_type, Np):
    """
    Make the given type of Ω_model based on number of particles
    """
    match Ω_model_type:
        case 'pairs':
            if Np == 1:
                Ω_first = 300*kHz
            elif Np <= 5:
                Ω_first = 325*kHz
            elif Np <= 10:
                Ω_first = 350*kHz
            return dict(Ω_model=Ω_model_type, Ω_first=Ω_first, Ω_interpair_gap=10*kHz, Ω_intrapair_gap=2*kHz)
        case 'equal spacing':
            return dict(Ω_model=Ω_model_type, Ω_mid=300*kHz, Ω_span=Np * 10*kHz)
        case 'custom':
            return dict(
                Ω_model=Ω_model_type,
                toy_Ωjn_dict={
                    dim_label: freq
                    for dim_label, freq in zip(
                        [
                            'x1', 'y1', 'x2', 'y2',
                            'x3', 'y3', 'x4', 'y4',
                            'x5', 'y5', 'x6', 'y6',
                            'x7', 'y7', 'x8', 'y8',
                        ],
                        np.array([
                            230.0, 232.5, 237.5, 240,
                            280, 282.5, 287.5, 290,
                            330.0, 332.5, 337.5, 340,
                            380, 382.5, 387.5, 390,
                        ]) * kHz,
                    )
                }
            )


def get_hd_plot_resolution_params(Ω_model_type, Np):
    """
    Get high resolution range bounds based on number of particles
    """
    match Ω_model_type:
        case 'pairs':
            if Np == 1:
                return 290, 310, 0.05
            elif Np <= 5:
                return 275, 335, 0.05
            elif Np <= 10:
                return 245, 355, 0.05
        case 'equal spacing':
            if Np == 1:
                return 290, 310, 0.01
            span = Np * 10
            return 300 - span / 2 - 5, 300 + span / 2 + 5, 0.05
        case 'custom':
            if Np == 1:
                return 220, 245, 0.05
            elif Np <= 4:
                return 220, 295, 0.05
            elif Np <= 6:
                return 220, 345, 0.05
            elif Np <= 8:
                return 220, 395, 0.05


def make_gjn_dict(Np, gx, gy):
    """
    Make a gjn_dict with given gx, gy values
    """
    gjn_dict = {}
    for ii in range(Np):
        gjn_dict[f'x{ii + 1}'] = gx
        gjn_dict[f'y{ii + 1}'] = gy
    return gjn_dict


def plot_vertical_stack_varying_1_parameter(
    system,
    Ω_model_type,
    parameter_to_vary,
    **kwargs,
):
    """
    Plot vertically stacked heterodyne spectra varying 1 parameter each time
    """
    Nd = system.split('×')
    Np, d = int(Nd[0]), int(Nd[1])

    gx = kwargs.get('gx', -40*kHz)

    match parameter_to_vary:
        case 'coupling':
            gjn_list = kwargs.get('gjn_list', -np.array([10, 20, 40, 80]) * kHz)
            gjn_dicts = [make_gjn_dict(Np, gx=g, gy=g) for g in gjn_list]
            descriptions = [f'$g/2\pi$ = {g / kHz:.0f} kHz' for g in gjn_list]
            colour = blue
        case 'gy/gx ratio':
            ratios = kwargs.get('ratios', [0.10, 0.25, 0.75, 1.00])
            gjn_dicts = [make_gjn_dict(Np, gx=gx, gy=ratio * gx) for ratio in ratios]
            descriptions = [f'$g_y/g_x$ = {ratio:.2f}' for ratio in ratios]
            colour = light_blue
        case 'detuning':
            Δ_list = kwargs.get('Δ_list', -np.array([900, 450, 225, 100]) * kHz)
            descriptions = [f'$\Delta/2\pi$ = {Δ / kHz:.0f} kHz' for Δ in Δ_list]
            colour = dark_yellow
        case 'optical damping':
            κ_list = kwargs.get('κ_list', np.array([800, 400, 200, 100]) * kHz)
            descriptions = [f'$\kappa/2\pi$ = {κ / kHz:.0f} kHz' for κ in κ_list]
            colour = purple
        case 'custom couplings':
            toy_gjn_list = kwargs.get('toy_gjn_list')
            descriptions = kwargs.get('descriptions')
            colour = green

    visualise_two_methods = kwargs.get('visualise_two_methods', False)
    if not visualise_two_methods:
        num_plots = len(descriptions)
        fig = plt.figure(figsize=(LATEX_FIG_WIDTH, 3 * num_plots), dpi=100)
        ax_dict = fig.subplot_mosaic([[f'plot{ii}'] for ii in range(num_plots)], sharex=True, gridspec_kw={'hspace': 0})

    Ω_model_dict = make_Ω_model_dict(Ω_model_type, Np)
    low, high, low_res = kwargs.get('plot_range_kHz', (0, 400 + 2.5, 2.5))
    hd_low, hd_high, hd_res = get_hd_plot_resolution_params(Ω_model_type, Np)
    ω_range = np.concatenate((
        np.arange(low, hd_low, low_res) * kHz,
        np.arange(hd_low, hd_high, hd_res) * kHz,
        np.arange(hd_high, high, low_res) * kHz,
    ))

    for ii, description in enumerate(descriptions):
        print(f'{description=}')
        gas_kwargs = dict(P=1e-7*mbar, T=300*K, vth=500*metre/second)
        gas_kwargs.update(kwargs.get('gas_kwargs_override', {}))

        cavity_kwargs = dict(
            Δ=-450*kHz if not parameter_to_vary == 'detuning' else Δ_list[ii],
            κ=200*kHz if not parameter_to_vary == 'optical damping' else κ_list[ii],
        )
        cavity_kwargs.update(kwargs.get('cavity_kwargs_override', {}))

        toy_gjn_dict = make_gjn_dict(Np, gx=gx, gy=gx)
        if parameter_to_vary in ['gy/gx ratio', 'coupling']:
            toy_gjn_dict = gjn_dicts[ii]

        toy_params = dict(gjn_dict=toy_gjn_dict)
        if parameter_to_vary == 'custom couplings':
            toy_params = dict(gjn_dict=toy_gjn_list[ii])
        toy_params.update(Ω_model_dict)

        mtrap_Nx2 = MultiTrap(
            num_traps=Np,
            Cartesian_axes_per_trap=d,
            ω_range=ω_range,
            gas_kwargs=gas_kwargs,
            cavity_kwargs=cavity_kwargs,
            toy_params=toy_params,
        )
        if visualise_two_methods:
            visualise_two_spectrum_calculation_methods(
                mtrap_Nx2, SpectrumType.HETERODYNE,
                method_A=QuantumLinearTheoryMethod.LINEAR_AMPLIFIER_MODEL,
                method_B=QuantumLinearTheoryMethod.HYBRID_LINEAR_AMPLIFIER_MODEL,
                title=f'\n{description}',
            )
        else:
            method = kwargs.get('method', QuantumLinearTheoryMethod.LINEAR_AMPLIFIER_MODEL)
            vals = mtrap_Nx2.heterodyne_spectrum(method)
            ax = ax_dict[f'plot{ii}']
            ax.fill_between(mtrap_Nx2.ω_range / kHz, vals, linewidth=1.5, alpha=0.8, edgecolor=colour, hatch='.....', facecolor='none')
            ax.set_yscale('log')

            ax.set_yticks([0.1, 1, 10, 100])
            ax.get_yaxis().set_major_formatter(plt.FormatStrFormatter('%.1f'))
            ax.set_ylim(1e-1, 0.99999 * 1e2)
            ax.set_xlim(min(ω_range) / kHz, max(ω_range) / kHz)

            textbox = dict(
                s=description, fontsize=15, ha='left', va='top', zorder=100,
                bbox=dict(facecolor=white, alpha=0.5, edgecolor='none', boxstyle='round,pad=0.2'),
            )
            ax.text(0.10, 0.85, transform=ax.transAxes, **textbox)

    if not visualise_two_methods:
        fig.suptitle(rf'$N_\mathregular{{p}}$={Np}; {Ω_model_type}', fontsize=20)
        ax_dict[f'plot{len(descriptions) - 1}'].set_xlabel('Frequency [kHz]', fontsize=15)
        fig.text(-0.05, 0.5, 'Heterodyne spectra [photons/Hz]', rotation=90, va='center', fontsize=15)
        plt.tight_layout()
        plt.show()



# Visualise two methods using custom (multiband) model

In [ ]:
system = '8×2'
plot_vertical_stack_varying_1_parameter(
    system,
    'custom',
    'detuning',
    Δ_list=-np.arange(100, 501, 100) * kHz,
    visualise_two_methods=True,
)


# Vary detuning on custom (multiband) model

In [ ]:
system = '8×2'
plot_vertical_stack_varying_1_parameter(
    system,
    'custom',
    'detuning',
    Δ_list=-np.arange(100, 1001, 100) * kHz,
)


# Vary RMS couplings on pairs and equal spacing models

In [ ]:
for system in [
    '2×2',
    '4×2',
    '8×2',
]:
    Nd = system.split('×')
    Np, d = int(Nd[0]), int(Nd[1])
    num_samples = Np * d
    lower_bound, upper_bound = 5, 100

    g_RMSs = [10, 20, 40, 60]
    descriptions = [r'$g_\mathregular{RMS} / 2\pi$' + f' = -{g_RMS} kHz' for g_RMS in g_RMSs]
    all_samples = {g_RMS: sample_with_RMS(num_samples, lower_bound, upper_bound, g_RMS) for g_RMS in g_RMSs}

    for samples in all_samples.values():
        print('Actual RMS:', np.sqrt(np.mean(samples**2)))
        print('Samples:', samples)

    def make_custom_gjn_dict(gjn_list, Np):
        """
        Make a gjn_dict that assigns the values of gjn_list in order
        """
        gjn_dict = {}
        iter_gjn_list = iter(gjn_list)
        for ii in range(Np):
            gjn_dict[f'x{ii + 1}'] = next(iter_gjn_list)
            gjn_dict[f'y{ii + 1}'] = next(iter_gjn_list)
        return gjn_dict

    toy_gjn_list = [make_custom_gjn_dict(-np.array(samples) * kHz, Np) for samples in all_samples.values()]

    plot_range_kHz = (0, 400 + 1, 1)
    plot_vertical_stack_varying_1_parameter(system, 'pairs', 'custom couplings', descriptions=descriptions, toy_gjn_list=toy_gjn_list, plot_range_kHz=plot_range_kHz)
    plot_vertical_stack_varying_1_parameter(system, 'equal spacing', 'custom couplings', descriptions=descriptions, toy_gjn_list=toy_gjn_list, plot_range_kHz=plot_range_kHz)



# All possibilities

In [ ]:
# takes about 1-2 min (60 figures)
for Ω_model_type in [
    'pairs',
    'equal spacing',
]:
    for system in [
        '1×2',
        '2×2',
        '4×2',
        '5×2',
        '8×2',
        '10×2',
    ]:
        for parameter_to_vary in [
            'coupling',
            'gy/gx ratio',
            'detuning',
            'optical damping',
        ]:
            plot_vertical_stack_varying_1_parameter(system, Ω_model_type, parameter_to_vary)

        plot_vertical_stack_varying_1_parameter(
            system, Ω_model_type, 'detuning',
            Δ_list=-np.linspace(150, 600, 4) * kHz,
            plot_range_kHz=(0, 600 + 2.5, 2.5),
        )

# END